In [0]:
 # Our Gold architecture:
 
 #                   SILVER
 #                      │
 #      ┌───────────────┼────────────────┐
 #      │               │                │
 #      ▼               ▼                ▼
 #    Demand          Supply          Inventory
 #      │               │                │
 #      ▼               ▼                ▼
 # Demand KPI       Supply KPI      Inventory KPI
 #      │               │                │
 #      └───────────────┼────────────────┘
 #                      │
 #             ┌────────▼─────────┐
 #             │ Transport / WMS  │
 #             │      KPIs         │
 #             └────────┬─────────┘
 #                      │
 #                      ▼
 #            ┌──────────────────┐
 #           │ Exception Engine │
 #           └────────┬─────────┘
 #                    ▼
 #         supply_chain_exceptions

In [0]:
# Gold tables
# workspace.gold
#│
#├── demand_kpi
#├── supply_kpi
#├── inventory_kpi
#├── transport_kpi
#├── warehouse_kpi
#└── supply_chain_exceptions

In [0]:
# Demand KPI
"""demand quantity
    order count
    material demand"""
# Supply KPI
"""PO quantity
   open quantity
   supplier/order metrics """
# Inventory KPI
"""on-hand inventory
   safety stock
   reorder point
   inventory value
   safety-stock breach"""
# Transport KPI
"""shipment count
   shipment status
   tracking status """
# Warehouse KPI
"""inventory
   picked quantity
   pending quantity"""


In [0]:
# Inspect the Silver business tables to understand column_names
# Inspect the Silver business tables

silver_tables = [
    "dim_material",
    "dim_plant",
    "dim_supplier",
    "fact_demand",
    "fact_supply",
    "fact_inventory",
    "fact_transport",
    "fact_warehouse"
]

for table_name in silver_tables:
    df = spark.table(f"workspace.silver.{table_name}")

    print("\n" + "=" * 70)
    print(f"{table_name}")
    print("=" * 70)

    print(df.columns)

In [0]:
for table_name in ["fact_demand", "fact_supply"]:
    df = spark.table(f"workspace.silver.{table_name}")

    print("\n" + "=" * 70)
    print(table_name)
    print("=" * 70)

    print(df.columns)

In [0]:
from pyspark.sql import functions as F


# ============================================================
# READ SILVER TABLES
# ============================================================

fact_demand = spark.table("workspace.silver.fact_demand")
fact_supply = spark.table("workspace.silver.fact_supply")
fact_inventory = spark.table("workspace.silver.fact_inventory")
fact_transport = spark.table("workspace.silver.fact_transport")
fact_warehouse = spark.table("workspace.silver.fact_warehouse")


# ============================================================
# 1. DEMAND KPI
# Grain: Material + Plant
# ============================================================

demand_kpi = (
    fact_demand
    .groupBy(
        "material_id",
        F.col("item_werks").alias("plant_id")
    )
    .agg(
        F.countDistinct("sales_order_id").alias("order_count"),

        F.sum(
            F.col("item_kwmeng").cast("double")
        ).alias("demand_qty"),

        F.sum(
            F.col("item_netwr").cast("double")
        ).alias("demand_value")
    )
    .withColumn(
        "avg_order_qty",
        F.when(
            F.col("order_count") > 0,
            F.col("demand_qty") / F.col("order_count")
        ).otherwise(0)
    )
)

demand_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.demand_kpi")


# ============================================================
# 2. SUPPLY KPI
# Grain: Material + Plant
# ============================================================

supply_base = (
    fact_supply
    .withColumn(
        "po_qty",
        F.col("item_menge").cast("double")
    )
    .withColumn(
        "open_qty",
        F.col("schedule_open_qty").cast("double")
    )
    .withColumn(
        "planned_delivery_date",
        F.to_date("schedule_eindt")
    )
)

supply_kpi = (
    supply_base
    .groupBy(
        "material_id",
        "plant_id"
    )
    .agg(
        F.countDistinct("po_id").alias("po_count"),

        F.sum("po_qty").alias("po_qty"),

        F.sum("open_qty").alias("open_po_qty"),

        F.sum(
            F.when(
                (F.col("planned_delivery_date") < F.current_date()) &
                (F.col("open_qty") > 0),
                1
            ).otherwise(0)
        ).alias("late_open_schedule_lines")
    )
    .withColumn(
        "supply_risk",
        F.when(
            F.col("open_po_qty") > 0,
            F.lit("OPEN_SUPPLY")
        ).otherwise(
            F.lit("NO_OPEN_SUPPLY")
        )
    )
)

supply_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.supply_kpi")


# ============================================================
# 3. INVENTORY KPI
# Grain: Material + Plant
# ============================================================

inventory_kpi = (
    fact_inventory
    .groupBy(
        "material_id",
        "plant_id"
    )
    .agg(
        F.sum(
            F.col("inventory_labst").cast("double")
        ).alias("unrestricted_stock"),

        F.sum(
            F.col("inventory_insme").cast("double")
        ).alias("quality_stock"),

        F.sum(
            F.col("inventory_speme").cast("double")
        ).alias("blocked_stock"),

        F.max(
            F.col("planning_safety_stock").cast("double")
        ).alias("safety_stock"),

        F.max(
            F.col("planning_reorder_point").cast("double")
        ).alias("reorder_point")
    )
    .withColumn(
        "inventory_vs_safety_stock",
        F.col("unrestricted_stock") -
        F.col("safety_stock")
    )
    .withColumn(
        "inventory_vs_reorder_point",
        F.col("unrestricted_stock") -
        F.col("reorder_point")
    )
    .withColumn(
        "inventory_risk",
        F.when(
            F.col("unrestricted_stock") < F.col("safety_stock"),
            F.lit("BELOW_SAFETY_STOCK")
        )
        .when(
            F.col("unrestricted_stock") < F.col("reorder_point"),
            F.lit("BELOW_REORDER_POINT")
        )
        .otherwise(
            F.lit("HEALTHY")
        )
    )
)

inventory_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.inventory_kpi")


# ============================================================
# 4. TRANSPORT KPI
# Grain: Origin Plant
# ============================================================

transport_base = (
    fact_transport
    .withColumn(
        "planned_delivery_date",
        F.to_date("shipment_planned_delivery_date")
    )
    .withColumn(
        "actual_delivery_date",
        F.to_date("shipment_actual_delivery_date")
    )
)

transport_kpi = (
    transport_base
    .groupBy(
        F.col("shipment_origin_plant").alias("plant_id")
    )
    .agg(
        F.countDistinct(
            "shipment_shipment_id"
        ).alias("shipment_count"),

        F.sum(
            F.when(
                F.col("shipment_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias("delivered_shipments"),

        F.sum(
            F.when(
                (
                    F.col("actual_delivery_date").isNotNull()
                ) &
                (
                    F.col("actual_delivery_date") >
                    F.col("planned_delivery_date")
                ),
                1
            ).otherwise(0)
        ).alias("delayed_shipments"),

        F.sum(
            F.when(
                (
                    F.col("actual_delivery_date").isNotNull()
                ) &
                (
                    F.col("actual_delivery_date") <=
                    F.col("planned_delivery_date")
                ),
                1
            ).otherwise(0)
        ).alias("on_time_shipments")
    )
    .withColumn(
        "on_time_delivery_pct",
        F.when(
            F.col("delivered_shipments") > 0,
            F.round(
                F.col("on_time_shipments") /
                F.col("delivered_shipments") * 100,
                2
            )
        ).otherwise(0)
    )
    .withColumn(
        "transport_risk",
        F.when(
            F.col("delayed_shipments") > 0,
            F.lit("TRANSPORT_DELAY")
        ).otherwise(
            F.lit("ON_TIME")
        )
    )
)

transport_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.transport_kpi")


# ============================================================
# 5. WAREHOUSE KPI
# Grain: Warehouse + Material
# ============================================================

warehouse_kpi = (
    fact_warehouse
    .groupBy(
        "inventory_warehouse_id",
        "inventory_material_id"
    )
    .agg(
        F.sum(
            F.col("inventory_on_hand_qty").cast("double")
        ).alias("on_hand_qty"),

        F.sum(
            F.col("inventory_allocated_qty").cast("double")
        ).alias("allocated_qty"),

        F.sum(
            F.col("inventory_available_qty").cast("double")
        ).alias("available_qty"),

        F.sum(
            F.col("pick_requested_qty").cast("double")
        ).alias("requested_pick_qty"),

        F.sum(
            F.col("pick_picked_qty").cast("double")
        ).alias("picked_qty")
    )
    .withColumn(
        "pending_pick_qty",
        F.col("requested_pick_qty") -
        F.col("picked_qty")
    )
    .withColumn(
        "warehouse_risk",
        F.when(
            F.col("pending_pick_qty") > 0,
            F.lit("PICK_PENDING")
        ).otherwise(
            F.lit("HEALTHY")
        )
    )
)

warehouse_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.warehouse_kpi")


# ============================================================
# 6. CROSS-FUNCTIONAL EXCEPTION ENGINE
# Grain: Material + Plant
# ============================================================

exceptions = (
    demand_kpi.alias("d")
    .join(
        supply_kpi.alias("s"),
        (
            (F.col("d.material_id") == F.col("s.material_id")) &
            (F.col("d.plant_id") == F.col("s.plant_id"))
        ),
        "left"
    )
    .join(
        inventory_kpi.alias("i"),
        (
            (F.col("d.material_id") == F.col("i.material_id")) &
            (F.col("d.plant_id") == F.col("i.plant_id"))
        ),
        "left"
    )
    .join(
        transport_kpi.alias("t"),
        F.col("d.plant_id") == F.col("t.plant_id"),
        "left"
    )
    .select(
        F.col("d.material_id"),
        F.col("d.plant_id"),

        F.col("d.demand_qty"),
        F.col("d.order_count"),

        F.coalesce(
            F.col("s.open_po_qty"),
            F.lit(0)
        ).alias("open_po_qty"),

        F.coalesce(
            F.col("s.late_open_schedule_lines"),
            F.lit(0)
        ).alias("late_open_schedule_lines"),

        F.col("i.unrestricted_stock"),
        F.col("i.safety_stock"),
        F.col("i.reorder_point"),

        F.coalesce(
            F.col("t.delayed_shipments"),
            F.lit(0)
        ).alias("delayed_shipments"),

        F.coalesce(
            F.col("t.on_time_delivery_pct"),
            F.lit(100)
        ).alias("on_time_delivery_pct")
    )
)


# ============================================================
# CREATE RISK FLAGS
# ============================================================

exceptions = (
    exceptions

    # Inventory risk
    .withColumn(
        "inventory_risk_flag",
        F.when(
            F.col("unrestricted_stock") <
            F.col("safety_stock"),
            1
        ).otherwise(0)
    )

    # Supply risk
    .withColumn(
        "supply_risk_flag",
        F.when(
            (
                F.col("open_po_qty") > 0
            ) &
            (
                F.col("late_open_schedule_lines") > 0
            ),
            1
        ).otherwise(0)
    )

    # Transport risk
    .withColumn(
        "transport_risk_flag",
        F.when(
            F.col("delayed_shipments") > 0,
            1
        ).otherwise(0)
    )

    # Demand risk
    .withColumn(
        "demand_risk_flag",
        F.when(
            F.col("demand_qty") > 0,
            1
        ).otherwise(0)
    )

    # Overall risk score
    .withColumn(
        "risk_score",
        F.col("inventory_risk_flag") +
        F.col("supply_risk_flag") +
        F.col("transport_risk_flag")
    )

    # Exception classification
    .withColumn(
        "exception_type",
        F.when(
            F.col("risk_score") >= 3,
            "CROSS_FUNCTIONAL_RISK"
        )
        .when(
            F.col("inventory_risk_flag") == 1,
            "INVENTORY_RISK"
        )
        .when(
            F.col("supply_risk_flag") == 1,
            "SUPPLY_RISK"
        )
        .when(
            F.col("transport_risk_flag") == 1,
            "TRANSPORT_RISK"
        )
        .otherwise(
            "NO_EXCEPTION"
        )
    )

    .withColumn(
        "exception_status",
        F.when(
            F.col("risk_score") > 0,
            "OPEN"
        ).otherwise(
            "NORMAL"
        )
    )

    .withColumn(
        "exception_created_date",
        F.current_date()
    )
)


# Keep only actual exceptions
supply_chain_exceptions = (
    exceptions
    .filter(F.col("risk_score") > 0)
)


supply_chain_exceptions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.supply_chain_exceptions")


# ============================================================
# FINAL OUTPUT
# ============================================================

print("==============================================")
print("GOLD LAYER COMPLETED")
print("==============================================")

print("Created:")
print("1. workspace.gold.demand_kpi")
print("2. workspace.gold.supply_kpi")
print("3. workspace.gold.inventory_kpi")
print("4. workspace.gold.transport_kpi")
print("5. workspace.gold.warehouse_kpi")
print("6. workspace.gold.supply_chain_exceptions")

print("==============================================")

In [0]:
%sql
SELECT
    'demand_kpi' AS table_name,
    COUNT(*) AS rows
FROM workspace.gold.demand_kpi

UNION ALL
SELECT
    'supply_kpi',
    COUNT(*)
FROM workspace.gold.supply_kpi

UNION ALL
SELECT
    'inventory_kpi',
    COUNT(*)
FROM workspace.gold.inventory_kpi

UNION ALL
SELECT
    'transport_kpi',
    COUNT(*)
FROM workspace.gold.transport_kpi

UNION ALL
SELECT
    'warehouse_kpi',
    COUNT(*)
FROM workspace.gold.warehouse_kpi

UNION ALL
SELECT
    'supply_chain_exceptions',
    COUNT(*)
FROM workspace.gold.supply_chain_exceptions;

In [0]:
%sql
SELECT *
FROM workspace.gold.supply_chain_exceptions
ORDER BY risk_score DESC
LIMIT 20;

In [0]:
%sql
SELECT
    exception_type,
    exception_status,
    COUNT(*) AS exception_count
FROM workspace.gold.supply_chain_exceptions
GROUP BY
    exception_type,
    exception_status
ORDER BY exception_count DESC;